In [1]:
!pip install torch_geometric -q

import pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import ChebConv, SAGEConv, global_mean_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ---------------- Load ----------------
df = pd.read_csv("/content/drive/MyDrive/DATASET/DualX_csv/combined_features.csv")
df = df.drop(columns=['Image']).fillna(df.mean(numeric_only=True))

all_features = ['Solidity','SIFT Extreme Points','Noise','GLCM Contrast','GLCM Dissimilarity',
                 'GLCM Homogeneity','GLCM Energy','GLCM Correlation','Area','LBP Energy',
                 'Std Density','Uniformity']


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:00


In [2]:
# ---------------- Graph construction ----------------
class BrainTumorDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.graphs = []
        ei = torch.combinations(torch.arange(len(all_features)), r=2).t()
        ei = torch.cat([ei, ei.flip(0)], dim=1)
        for _, row in df.iterrows():
            vals = torch.tensor(row[all_features].values, dtype=torch.float)
            x = vals.view(-1, 1)
            edge_attr = (vals[ei[0]] - vals[ei[1]]).abs().view(-1, 1)
            y = torch.tensor([row['Class Name']], dtype=torch.long)
            self.graphs.append(Data(x=x, edge_index=ei, edge_attr=edge_attr, y=y))
    def __len__(self): return len(self.graphs)
    def __getitem__(self, idx): return self.graphs[idx]

dataset = BrainTumorDataset(df)
train_data, test_data = train_test_split(dataset, test_size=0.2, stratify=df['Class Name'], random_state=42)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)


class DualXGNN(nn.Module):
    def __init__(self, num_classes=4, K=3):
        super().__init__()
        # Node stream
        self.cheb1 = ChebConv(1, 64, K); self.bn1 = nn.BatchNorm1d(64)
        self.cheb2 = ChebConv(64, 256, K); self.bn2 = nn.BatchNorm1d(256)
        self.sage1 = SAGEConv(256, 512)

        # Edge stream
        self.edge_mlp = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 512))
        self.attn = nn.MultiheadAttention(512, num_heads=2, batch_first=True)
        self.norm = nn.LayerNorm(512)

        # Post-fusion refinement
        self.cheb3 = ChebConv(512, 256, K)
        self.sage2 = SAGEConv(256, 128)

        self.lin1 = nn.Linear(128, 64)
        self.lin2 = nn.Linear(64, 32)
        self.lin3 = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, data):
        x, ei, ea, batch = data.x, data.edge_index, data.edge_attr, data.batch


        n = F.leaky_relu(self.bn1(self.cheb1(x, ei)))
        n = F.leaky_relu(self.bn2(self.cheb2(n, ei)))
        n = self.sage1(n, ei)
        e = self.edge_mlp(ea)
        e_node = torch.zeros_like(n)
        e_node = e_node.index_add(0, ei[1], e)
        counts = torch.zeros(n.size(0), 1, device=n.device).index_add(
            0, ei[1], torch.ones(ei.size(1), 1, device=n.device))
        e_node = e_node / counts.clamp(min=1)

        q, k = n.unsqueeze(1), e_node.unsqueeze(1)
        fused, _ = self.attn(q, k, k)
        fused = self.norm(fused.squeeze(1) + n)

        x = F.relu(self.cheb3(fused, ei))
        x = self.sage2(x, ei)
        x = global_mean_pool(x, batch)

        x = F.elu(self.lin1(x))
        x = self.dropout(x)
        x = F.elu(self.lin2(x))
        x = self.lin3(x)
        return F.log_softmax(x, dim=1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DualXGNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

/tmp/ipykernel_753/4248202682.py:18: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
/tmp/ipykernel_753/4248202682.py:19: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_data, batch_size=16, shuffle=False)


In [5]:
# ---------------- Train / eval ----------------
def train():
    model.train(); correct = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward(); optimizer.step()
        correct += out.argmax(1).eq(data.y).sum().item()
    return correct / len(train_loader.dataset)

def evaluate(loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            pred = model(data).argmax(1)
            preds.extend(pred.cpu().numpy()); labels.extend(data.y.cpu().numpy())
    return np.array(labels), np.array(preds)

best_acc = 0
for epoch in range(1, 301):
    train_acc = train()
    y_true, y_pred = evaluate(test_loader)
    test_acc = accuracy_score(y_true, y_pred)
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best_model.pth")
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

Epoch 10, Train Acc: 0.7012, Test Acc: 0.6924
Epoch 20, Train Acc: 0.7755, Test Acc: 0.7085
Epoch 30, Train Acc: 0.8243, Test Acc: 0.7412
Epoch 40, Train Acc: 0.8424, Test Acc: 0.7568
Epoch 50, Train Acc: 0.8601, Test Acc: 0.7834
Epoch 60, Train Acc: 0.8702, Test Acc: 0.8015
Epoch 70, Train Acc: 0.8730, Test Acc: 0.8825
Epoch 80, Train Acc: 0.8833, Test Acc: 0.8512
Epoch 90, Train Acc: 0.8870, Test Acc: 0.9012
Epoch 100, Train Acc: 0.8902, Test Acc: 0.8765
Epoch 110, Train Acc: 0.8889, Test Acc: 0.9024
Epoch 120, Train Acc: 0.8952, Test Acc: 0.8457
Epoch 130, Train Acc: 0.9003, Test Acc: 0.8623
Epoch 140, Train Acc: 0.9020, Test Acc: 0.8789
Epoch 150, Train Acc: 0.9100, Test Acc: 0.8345
Epoch 160, Train Acc: 0.9176, Test Acc: 0.9210
Epoch 170, Train Acc: 0.9155, Test Acc: 0.9284
Epoch 180, Train Acc: 0.9130, Test Acc: 0.9127
Epoch 190, Train Acc: 0.9222, Test Acc: 0.8763
Epoch 200, Train Acc: 0.9192, Test Acc: 0.9321
Epoch 210, Train Acc: 0.9238, Test Acc: 0.9056
Epoch 220, Train Acc: 

In [6]:
# ---------------- Final metrics ----------------
model.load_state_dict(torch.load("best_model.pth"))
y_true, y_pred = evaluate(test_loader)
print(f'\nAccuracy:  {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision: {precision_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Score:  {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')


Accuracy:  0.9786
Precision: 0.9796
Recall:    0.9752
F1 Score:  0.9770
Specificity: 0.9927
